# పాఠం 13 - ఏజెంట్ మేమరీ


## Setup

This notebook demonstrates how to build a travel booking agent with **persistent memory** using the **Microsoft Agent Framework** (MAF).

You will learn how different types of agent memory — working, short-term, and long-term — affect how an agent retains and uses information across conversations.

**Prerequisites:**
- A Microsoft Foundry project with a deployed chat model (e.g. `gpt-5-mini`).
- Logged in with the Azure CLI — run `az login` in your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ఏజెంట్ జ్ఞాపకశక్తి రకాలూ

AI ఏజెంట్లు వివిధ రకాల జ్ఞాపకశక్తిని ఉపయోగించుకోవచ్చు, ప్రతి ఒక్కటి ఒక ప్రత్యేక ప్రయోజనాన్ని అందిస్తుంది:

### వర్కింగ్ మెమరీ
సంభాషణ థ్రెడ్ స్వయంగా — ఒకే సెషన్‌లో మార్పిడి చేసిన సందేశాలు. ఏజెంట్ సూటికాకుండా ఉండడానికి అదే థ్రెడ్‌లోని ముందు సందేశాల వైపు తిరిగి చూడొచ్చు. MAF లో ఇది **`agent.create_session()`** ద్వారా సృష్టించబడుతుంది, ఇది ఒక `AgentSession` ను ఇస్తుంది.

### షార్ట్-టర్మ్ మెమరీ
ఒక పని లేదా సెషన్ వ్యవధి పాటు నిలిచే సమాచారము కానీ శాశ్వతంగా నిల్వ చేయరాదు. ఉదాహరణకు, ఏజెంట్ బహుళ తాళాల ప్రణాళిక సంభాషణలో నిజాల్ని సేకరించి వాటిని చివరి ప్రయాణవ్యవస్థాపన కోసం ఉపయోగించవచ్చు.

### లాంగ్-టర్మ్ మెమరీ
సెషన్ల మధ్య నిల్వ ఉండే ప్రాధాన్యతలు మరియు నిజాలు. తిరిగి వచ్చే వినియోగదారుడు తన ఆహార పరిమితులు లేదా ప్రయాణ శైలిని మళ్ళీ చెప్పాల్సిన అవసరం లేదు. లాంగ్-టర్మ్ మెమరీ సాధారణంగా బాహ్య నిల్వ ద్వారా — డేటాబేస్, ఫైల్ లేదా వెక్టర్ ఇండెక్స్ — మద్దతు పొందుతుంది మరియు టూల్స్ ద్వారా ఏజెంట్‌కు ప్రదర్శించబడుతుంది.


## సెషన్లతో పని చేసే మెమరీ

మెమరీ యొక్క సర్వసాధారణ రూపం సంభాషణ సెషన్. మీరు అదే సెషన్ ( `agent.create_session()` ద్వారా సృష్టించబడింది) ను వరుసగా `agent.run()` పిలుపులకు ఇవ్వగలిగితే, ఏజెంట్ ఆ సంభాషణ యొక్క పూర్తి చరిత్రను చూస్తుంది మరియు పూర్వ వివరాలను గుర్తు చేసుకోవచ్చు.

ఒక ప్రయాణ ఏజెంట్ సృష్టించి పని చేసే మెమరీని ప్రదర్శిద్దాం.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ఏజెంట్ బడ్జెట్‌ను సరిగ్గా గుర్తుచేశాడు ఎందుకంటే రెండు సందేశాలు ఒకే సెషన్‌ను పంచుకున్నాయి. ఇది **వర్కింగ్ మెల్లినీరు** — ఇది కేవలం సెషన్ జీవితం కోసం మాత్రమే ఉంటుంది.

### కొత్త థ్రెడ్‌తో ఏమి జరుగుతుంది?

మేము **కొత్త** సెషన్‌ను సృష్టిస్తే, ఏజెంట్‌కు గత సంభాషణ యొక్క జ్ఞాపకం ఉంటుంది లేదు:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## దీర్ఘకాలిక జ్ఞాపక శైలి

వాడుకరి అభిరుచులను **సెషన్ల మధ్య** గుర్తుంచుకోవడానికి, సంభాషణ థ్రెడ్ బయట ఉండే శాశ్వత నిల్వ అవసరం. ఏజెంట్ ఈ నిల్వను **ఉపకరణాలు** ద్వారా అనుసంధానం చేస్తుంది — సమాచారాన్ని సేవ్ చేయడానికి మరియు పునఃప్రాప్తి చేసుకోవడానికి ఇది కాల్ చేయగల ఫంక్షన్లు.

క్రింద మేము ఒక సాధారణ మెమరీలోని అభిరుచుల నిల్వను అమలు చేస్తున్నాము (ఉత్పత్తిలో మీరు దీన్ని డేటాబేస్ లేదా వెక్టార్ సూచికతో మద్దతు చేస్తారు) మరియు దీన్ని ఏజెంట్ ఉపయోగించగల ఉపకరణాలుగా అందిస్తున్నాము.

### నిర్మాణ విధానం
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### దృశ్యం 1 — మొదటిసారి ఉపయోగించే వ్యక్తి వార్షికోత్సవ యాత్ర బుక్ చేసుకుంటాడు

సారా మొదటిసారిగా సందర్శించుతుంది. ఏజెంట్ ఆమె ఇష్టాలను టూల్స్ ద్వారా భద్రపరచి, వాటిని హోటెల్స్‌ని సిఫారసు చేయడంలో ఉపయోగించాలి.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### పరిస్థితి 2 — సారా కొన్ని వారం లలో తిరిగి వస్తుంది

సారా ఒక **కొత్త థ్రెడ్** ను ప్రారంభిస్తుంది (కొత్త సెషన్ ని అనుకరించటం). వర్కింగ్ మెమరీ ఖాళీగా ఉంటుంది, కానీ దీర్ఘకాలిక ప్రాధాన్యత నిల్వలో ఆమె సమాచారం ఇంకా ఉంది. ఏజెంట్ దాన్ని తీసుకుని వ్యక్తిపరసిపరణ సిఫార్సులను చేయాలి.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## సారాంశం

ఈ పాఠంలో మీరు మూడు రకాల ఏజెంట్ మెమరీలు మరియు వాటిని Microsoft Agent Framework తో ఎలా అమలు చేయాలో నేర్చుకున్నారు:

| మెమరీ రకం | MAF యంత్రాంగం | జీవించు కాలం |
|---|---|---|
| **వర్కింగ్** | `agent.create_session()` | ఒక సమావేశం మాట్లాడటం |
| **షార్ట్-టర్మ్** | ఒక థ్రెడ్ లో కూడిన సందర్భం | ఒక పని / సమావేశం |
| **లాంగ్-టర్మ్** | `@tool` ఫంక్షన్ల ద్వారా పరిక్షేత్ర నిల్వ | సమావేశాలపైనా |

### ముఖ్యమైన విషయాలు
1. **`agent.create_session()`** వర్కింగ్ మెమరీ ఇస్తుంది — ఏజెంట్ ఓ సెషన్ లో పూర్తి సంభాషణ చరిత్రను చూడగలడు.
2. **కొత్త సెషన్లు సందర్భం కోల్పోతాయి** — లాంగ్-టర్మ్ మెమరీ లేకపోతే ఏజెంట్ పాత సంభాషణలను గుర్తు చేయలేకపోతుంది.
3. **`@tool` ఫంక్షన్లు** మధ్య దూరం భర్తీ చేస్తాయి — అవి ఏజెంట్ కి సమాచారం నిల్వ చేసి తిరిగి పొందడానికి సహాయపడతాయి.
4. **వ్యక్తిగతీకరణ కాలంతో మెరుగుపడుతుంది** — ఎక్కువ ప్రాధాన్యాలు నిల్వ అయితే, ఏజెంట్ సూచనలు మరింత మెరుగుపడతాయి.

### వాస్తవ లో ప్రపంచ అనువర్తనలు
- **కస్టమర్ సర్వీస్**: కస్టమర్ చరిత్ర మరియు ప్రాధాన్యాలను గుర్తుంచు
- **వ్యక్తిగత సహాయకులు**: రోజులూ వారాలుగా సందర్భాన్ని నిర్వహించు
- **ఆరోగ్యం**: రోగి సమాచారం మరియు ప్రాధాన్యాలను ట్రాక్ చేయు
- **ఈ-కామర్స్**: చరిత్ర ఆధారంగా వ్యక్తిగత కొనుగోలు

### తరువాతి దశలు
- ఇన్-మెమరీ డిక్ట్ ని డేటాబేస్ లేదా వెక్టర్ స్టోర్ (ఉదా. Azure AI Search) తో మార్చు
- సమయ సున్నితమైన సమాచారానికి మెమరీ అవధి చేర్పు
- పంచుకున్న మెమరీతో బహు-ఏజెంట్ సిస్టమ్స్ నిర్మాణం
- జ్ఞాº-గ్రాఫ్ మద్దతు మెమరీ కోసం Cognee నోట్బుక్ ని అన్వేషించు


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
